In [1]:
import os
import gc
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [2]:
files = {
    10: "/kaggle/input/datasets/chetannarware/pemsbay/pemsbaay/pemsbay_10percent_missing.csv",
    20: "/kaggle/input/datasets/chetannarware/pemsbay/pemsbaay/pemsbay_20percent_missing.csv",
    40: "/kaggle/input/datasets/chetannarware/pemsbay/pemsbaay/pemsbay_40percent_missing.csv",
    80: "/kaggle/input/datasets/chetannarware/pemsbay/pemsbaay/pemsbay_80percent_missing.csv",
     0: "/kaggle/input/datasets/chetannarware/pemsbay/pemsbaay/pemsbay_0percent_missing.csv"
}
print(files[10])

/kaggle/input/datasets/chetannarware/pemsbay/pemsbaay/pemsbay_10percent_missing.csv


In [3]:
def create_sequences(data, mask, seq_len=10):
    X, M, Y = [], [], []
    for i in range(len(data) - seq_len):
        X.append(data[i:i+seq_len])
        M.append(mask[i:i+seq_len])
        Y.append(data[i+seq_len])
    return np.array(X), np.array(M), np.array(Y)

In [4]:
def clear_memory():
    gc.collect()
    torch.cuda.empty_cache()

In [5]:
class IRNN(nn.Module):
    """
    RNN with Imputation Unit.
      - Imputation uses h (RNN has no cell state)
      - Mask fed as extra input to gates
      - Returns imputations for regularization loss
      - No activation on imputation — full N(0,1) range
      - Uses LAST hidden state for prediction (no attention)
    """
    def __init__(self, input_dim, hidden_dim=64):
        super().__init__()
        self.hidden_dim = hidden_dim

        # Imputation unit: infers x̃_t from h only
        self.impute_net = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim)
        )

        # RNNCell: input is x_prime + mask = 2 * input_dim
        self.rnn_cell = nn.RNNCell(
            input_size=2 * input_dim,
            hidden_size=hidden_dim
        )

        # Output head
        self.output_layer = nn.Linear(hidden_dim, input_dim)

    def forward(self, x, mask):
        """
        x    : (B, T, F)
        mask : (B, T, F)
        Returns:
          pred        : (B, F)
          imputations : (B, T, F)
        """
        B, T, F = x.shape

        h_t = torch.zeros(B, self.hidden_dim, device=x.device)

        imputations = []

        for t in range(T):
            xt = x[:, t, :]
            mt = mask[:, t, :]

            # Imputation from h only
            x_tilde = self.impute_net(h_t)

            # Mask fusion
            x_prime = mt * xt + (1.0 - mt) * x_tilde

            imputations.append(x_tilde)

            # Feed x_prime + mask to RNN
            x_input = torch.cat([x_prime, mt], dim=1)
            h_t = self.rnn_cell(x_input, h_t)

        # Last hidden state only
        pred = self.output_layer(h_t)

        imputations = torch.stack(imputations, dim=1)

        return pred, imputations

In [6]:
def compute_loss(pred, target, imputations, x_input, mask, lam=0.1):
    """
    Same loss as IConvLSTM — ensures fair comparison.
    Prediction loss + imputation regularization on observed positions.
    """
    pred_loss = torch.mean(torch.abs(pred - target)) + \
                0.1 * torch.mean((pred - target) ** 2)

    imp_error = torch.abs(imputations - x_input) * mask
    imp_loss  = imp_error.sum() / (mask.sum() + 1e-8)

    return pred_loss + lam * imp_loss

In [7]:
def train_model(model, train_loader, val_loader, percent,
                max_epochs=80, patience=15):
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-5
    )

    best_loss = float("inf")
    wait = 0
    best_path = f"/kaggle/working/best_{percent}.pt"

    for epoch in range(max_epochs):
        model.train()
        train_loss = 0.0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{max_epochs}", leave=False)
        for x, m, y in pbar:
            x, m, y = x.to(device), m.to(device), y.to(device)
            optimizer.zero_grad()
            pred, imputations = model(x, m)
            loss = compute_loss(pred, y, imputations, x, m, lam=0.1)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item()
            pbar.set_postfix(loss=f"{loss.item():.4f}")

        train_loss /= len(train_loader)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for x, m, y in val_loader:
                x, m, y = x.to(device), m.to(device), y.to(device)
                pred, _ = model(x, m)
                val_loss += (torch.mean(torch.abs(pred - y)) +
                             0.1 * torch.mean((pred - y) ** 2)).item()
        val_loss /= len(val_loader)

        current_lr = optimizer.param_groups[0]['lr']
        scheduler.step(val_loss)
        new_lr = optimizer.param_groups[0]['lr']
        lr_msg = f"  → LR dropped to {new_lr:.2e}" if new_lr < current_lr else ""

        print(f"Epoch {epoch+1:02d} | Train: {train_loss:.4f} | "
              f"Val: {val_loss:.4f} | LR: {current_lr:.2e}{lr_msg}")

        if val_loss < best_loss:
            best_loss = val_loss
            wait = 0
            torch.save({"model": model.state_dict()}, best_path)
            print("  ✓ Saved best model")
        else:
            wait += 1
            if wait >= patience:
                print("  Early stopping triggered")
                break

    state = torch.load(best_path)["model"]
    model.load_state_dict(state)
    return model

In [8]:
def evaluate_model(model, loader, mean, std):
    model.eval()
    preds, trues, masks = [], [], []

    with torch.no_grad():
        for x, m, y in loader:
            x, m = x.to(device), m.to(device)
            pred, _ = model(x, m)
            preds.append(pred.cpu().numpy())
            trues.append(y.numpy())
            masks.append(m[:, -1, :].cpu().numpy())

    preds = np.concatenate(preds)
    trues = np.concatenate(trues)
    masks = np.concatenate(masks)

    mae_norm  = mean_absolute_error(trues.ravel(), preds.ravel())
    rmse_norm = np.sqrt(mean_squared_error(trues.ravel(), preds.ravel()))

    preds_real = preds * std + mean
    trues_real = trues * std + mean

    mae_real  = mean_absolute_error(trues_real.ravel(), preds_real.ravel())
    rmse_real = np.sqrt(mean_squared_error(trues_real.ravel(), preds_real.ravel()))

    r2_list = [r2_score(trues_real[:, i], preds_real[:, i])
               for i in range(trues_real.shape[1])]
    r2_real = np.mean(r2_list)

    valid = (np.abs(trues_real) > 1e-3) & (masks == 1)
    mape  = np.mean(np.abs((trues_real[valid] - preds_real[valid]) /
                            trues_real[valid])) * 100

    print("\n==============================")
    print("Normalized Results")
    print("==============================")
    print(f"MAE  : {mae_norm:.4f}")
    print(f"RMSE : {rmse_norm:.4f}")
    print("\n==============================")
    print("Denormalized Results (Real Scale)")
    print("==============================")
    print(f"MAE  : {mae_real:.4f}")
    print(f"RMSE : {rmse_real:.4f}")
    print(f"R2   : {r2_real:.4f}")
    print(f"MAPE : {mape:.2f}%")

    return mae_real, rmse_real, r2_real, mape

In [9]:
def train_single_dataset_irnn(percent, model_name="irnn"):
    print(f"\n{'='*30}\nSTARTING EXPERIMENT: {percent}% MISSING\n{'='*30}")

    raw = pd.read_csv(files[percent])
    raw = raw.apply(pd.to_numeric, errors='coerce')

    mask = (~raw.isna()).astype(np.float32).values
    data = raw.values.astype(np.float32)

    mask[:200, :] = 1.0
    col_medians = np.nanmedian(data, axis=0)
    col_medians = np.where(np.isnan(col_medians), 0.0, col_medians)
    for col in range(data.shape[1]):
        nan_rows = np.isnan(data[:200, col])
        if nan_rows.any():
            data[:200, col][nan_rows] = col_medians[col]

    split = int(0.8 * len(data))
    train_data, val_data = data[:split],  data[split:]
    train_mask, val_mask = mask[:split],  mask[split:]

    obs_train = train_data.copy()
    obs_train[train_mask == 0] = np.nan
    mean = np.nanmean(obs_train, axis=0, keepdims=True)
    std  = np.nanstd(obs_train,  axis=0, keepdims=True)
    mean = np.where(np.isnan(mean), 0.0, mean)
    std  = np.where(np.isnan(std) | (std < 1e-8), 1.0, std)

    print(f"Mean (first 5): {mean[0, :5]}")
    print(f"Std  (first 5): {std[0, :5]}")

    train_norm = np.where(train_mask == 1, (train_data - mean) / std, 0.0)
    val_norm   = np.where(val_mask   == 1, (val_data   - mean) / std, 0.0)

    SEQ_LEN = 10
    X_tr, M_tr, Y_tr = create_sequences(train_norm, train_mask, SEQ_LEN)
    X_vl, M_vl, Y_vl = create_sequences(val_norm,   val_mask,   SEQ_LEN)

    train_loader = DataLoader(
        TensorDataset(
            torch.tensor(X_tr, dtype=torch.float32),
            torch.tensor(M_tr, dtype=torch.float32),
            torch.tensor(Y_tr, dtype=torch.float32)
        ),
        batch_size=64, shuffle=True,  num_workers=0
    )
    val_loader = DataLoader(
        TensorDataset(
            torch.tensor(X_vl, dtype=torch.float32),
            torch.tensor(M_vl, dtype=torch.float32),
            torch.tensor(Y_vl, dtype=torch.float32)
        ),
        batch_size=64, shuffle=False, num_workers=0
    )

    input_dim = X_tr.shape[2]
    model = IRNN(input_dim=input_dim, hidden_dim=64).to(device)
    print(f"Using device: {device}")
    print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

    model = train_model(model, train_loader, val_loader, percent)

    save_path = f"/kaggle/working/{model_name}_{percent}.pt"
    torch.save({"model": model.state_dict(), "mean": mean, "std": std}, save_path)
    print("Saved:", save_path)

    mae, rmse, r2, mape = evaluate_model(model, val_loader, mean, std)
    return mae, rmse, r2, mape

In [10]:
results_rnn = {}

RUN_PERCENTS = [0, 20, 40, 80]

for RUN_PERCENT in RUN_PERCENTS:
    print(f"\n{'='*10} {RUN_PERCENT}% Missing {'='*10}")
    
    results_rnn[RUN_PERCENT] = train_single_dataset_irnn(
        RUN_PERCENT, model_name="pemsbay_irnn"
    )
    
    mae, rmse, r2, mape = results_rnn[RUN_PERCENT]

    print(f"\nFINAL RESULT {RUN_PERCENT}%")
    print(f"MAE  : {mae:.4f}")
    print(f"RMSE : {rmse:.4f}")
    print(f"R2   : {r2:.4f}")
    print(f"MAPE : {mape:.2f}%")

    clear_memory()


# ===== FINAL SUMMARY =====
print("\n" + "="*30)
print("ALL RESULTS SUMMARY")
print("="*30)

for p in RUN_PERCENTS:
    mae, rmse, r2, mape = results_rnn[p]
    print(f"\n{p}% Missing:")
    print(f"MAE  : {mae:.4f}")
    print(f"RMSE : {rmse:.4f}")
    print(f"R2   : {r2:.4f}")
    print(f"MAPE : {mape:.2f}%")


========== 0% Missing ==========

STARTING EXPERIMENT: 0% MISSING


/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:1233: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,


Mean (first 5): [ 0.       67.53788  59.032124 59.0796   62.214527]
Std  (first 5): [ 1.        8.261381 13.256112 11.864664  8.572646]
Using device: cuda
Parameters: 92,492


Epoch 1/80:   0%|          | 0/652 [00:00<?, ?it/s]

Epoch 01 | Train: 0.3817 | Val: 0.3401 | LR: 1.00e-03
  ✓ Saved best model


...

Epoch 78 | Train: 0.1750 | Val: 0.1519 | LR: 1.00e-05


Epoch 79/80:   0%|          | 0/652 [00:00<?, ?it/s]

Epoch 79 | Train: 0.1749 | Val: 0.1519 | LR: 1.00e-05


Epoch 80/80:   0%|          | 0/652 [00:00<?, ?it/s]

Epoch 80 | Train: 0.1750 | Val: 0.1519 | LR: 1.00e-05
Saved: /kaggle/working/pemsbay_irnn_80.pt

Normalized Results
MAE  : 0.1295
RMSE : 0.4742

Denormalized Results (Real Scale)
MAE  : 1.0813
RMSE : 4.1798
R2   : -0.0046
MAPE : 2.82%

FINAL RESULT 80%
MAE  : 1.0813
RMSE : 4.1798
R2   : -0.0046
MAPE : 2.82%

ALL RESULTS SUMMARY

0% Missing:
MAE  : 1.8917
RMSE : 3.6607
R2   : 0.7634
MAPE : 4.24%

20% Missing:
MAE  : 2.5076
RMSE : 4.9843
R2   : 0.5817
MAPE : 5.29%

40% Missing:
MAE  : 2.8894
RMSE : 5.4719
R2   : 0.3879
MAPE : 6.19%

80% Missing:
MAE  : 1.0813
RMSE : 4.1798
R2   : -0.0046
MAPE : 2.82%
